# Goodreads Genre Trends — Relational Database Design

**Capstone requirement coverage:**
1. Relational database design and reasoning
2. Entity-Relationship Diagram (ERD)
3. Database build in SQLite3 using Python
4. Intermediate/advanced SQL queries (joins, aggregation, `HAVING`, subqueries)

## 1. Database Design & Reasoning

The source data (Goodreads books scraped by genre) starts out flat: one row per book per genre tag, with author names repeated as free text on every row. This works fine for pandas-based EDA, but it violates normalization if stored directly in a relational database:

- **Author names are repeated** across every book they wrote, and across every genre-tag row for that book. This wastes space and risks inconsistency (e.g. `"J.K. Rowling"` vs `"JK Rowling"` being treated as two different authors).
- **Genre labels are repeated** thousands of times as raw strings rather than being a controlled, referenceable set of values.
- **Books and genres have a many-to-many relationship.** Exploratory analysis on this dataset showed books are typically tagged with 15+ genres each. A single foreign key on either the `books` or `genres` table cannot represent this — it requires a **junction (bridge) table**.

### Proposed schema (3rd Normal Form)

| Table | Purpose | Key columns |
|---|---|---|
| `authors` | One row per unique author | `author_id` (PK), `author_name` |
| `books` | One row per unique book (the core fact table) | `book_id` (PK), `name`, `author_id` (FK), `pub_year`, `star_rating`, `num_ratings`, `isbn_clean` |
| `genres` | One row per unique genre label | `genre_id` (PK), `genre_name` |
| `book_genres` | Junction table resolving the books↔genres many-to-many relationship | `book_id` (FK), `genre_id` (FK), composite PK |

### Why this design

- **Normalization eliminates redundant text storage.** Author and genre names are stored once and referenced by integer key everywhere else, which is smaller and enforces consistency (a genre can't be spelled two different ways across rows).
- **The junction table correctly models many-to-many.** `book_genres` allows any book to have any number of genres and any genre to apply to any number of books, without duplicating book or genre data.
- **Referential integrity.** Foreign keys mean a `book_genres` row can't point to a nonexistent book or genre, and a book can't be linked to a nonexistent author.
- **Query efficiency.** Questions like "most prevalent genre per year" or "which genres co-occur with History" become straightforward joins/aggregations instead of string-matching across a flat table, and can be indexed for performance at scale (millions of rows).

## 2. Entity-Relationship Diagram (ERD)

```
AUTHORS ||--o{ BOOKS        : writes
BOOKS   ||--o{ BOOK_GENRES  : tagged_with
GENRES  ||--o{ BOOK_GENRES  : applied_to

AUTHORS
  author_id   INTEGER  PK
  author_name TEXT

BOOKS
  book_id      INTEGER  PK
  name         TEXT
  author_id    INTEGER  FK -> AUTHORS.author_id
  pub_year     INTEGER
  star_rating  REAL
  num_ratings  INTEGER
  isbn_clean   TEXT

GENRES
  genre_id    INTEGER  PK
  genre_name  TEXT

BOOK_GENRES
  book_id   INTEGER  FK -> BOOKS.book_id
  genre_id  INTEGER  FK -> GENRES.genre_id
  (composite PK: book_id + genre_id)
```

**Relationships:**
- `authors` 1---to---many `books` (one author can write many books)
- `books` many---to---many `genres`, resolved through `book_genres`

*(An interactive ERD was also rendered in the accompanying chat session — reproduce with `erDiagram` syntax in any Mermaid-compatible renderer, e.g. [mermaid.live](https://mermaid.live), using the schema above.)*

## 3. Build the Database in SQLite3 with Python

In [4]:
import sqlite3
import pandas as pd
from pathlib import Path
import pyarrow.parquet as pq
import pyarrow as pa
import numpy as np


conn = sqlite3.connect("goodreads.db")
cur = conn.cursor()

# Enable foreign key enforcement (off by default in SQLite)
cur.execute("PRAGMA foreign_keys = ON;")

folder = Path("Data/Goodreads_Books/genres_top100")
keep_cols = ["name", "author", "genres", "pub_year", "star_rating", "num_ratings", "isbn_clean"]

tables = []
for file in folder.glob("*.parquet"):
    table = pq.read_table(file, columns=keep_cols)
    genre_col = pa.array([file.stem] * table.num_rows)
    table = table.append_column("source_genre", genre_col)
    tables.append(table)

combined_table = pa.concat_tables(tables, promote_options="default")
df = combined_table.to_pandas()

df["pub_year"] = pd.to_numeric(df["pub_year"], errors="coerce", downcast="integer")
df["source_genre"] = df["source_genre"].astype("category")

# Filter to reliable year range
valid = df[(df["pub_year"] >= 1900) & (df["pub_year"] <= 2016)].copy()

# Build genres_list
valid["genres_list"] = valid["genres"].apply(
    lambda g: [str(x).strip() for x in g] if isinstance(g, (list, tuple, np.ndarray)) else []
)

# Explode so each row is one (book, genre) pair
exploded = valid.explode("genres_list")
exploded = exploded[exploded["genres_list"].notna() & (exploded["genres_list"] != "")]
exploded["genres_list"] = exploded["genres_list"].str.strip().str.lower()

# Split out fiction/non-fiction as a category, not a genre
fiction_labels = {"fiction", "non-fiction", "nonfiction"}
genre_only = exploded[~exploded["genres_list"].isin(fiction_labels)]

print(valid.shape, genre_only.shape)

(4352106, 9) (24100598, 9)


In [5]:
# --- Create tables ---
cur.executescript("""
CREATE TABLE IF NOT EXISTS authors (
    author_id INTEGER PRIMARY KEY AUTOINCREMENT,
    author_name TEXT NOT NULL UNIQUE
);

CREATE TABLE IF NOT EXISTS books (
    book_id INTEGER PRIMARY KEY AUTOINCREMENT,
    name TEXT NOT NULL,
    author_id INTEGER,
    pub_year INTEGER,
    star_rating REAL,
    num_ratings INTEGER,
    isbn_clean TEXT,
    FOREIGN KEY (author_id) REFERENCES authors(author_id)
);

CREATE TABLE IF NOT EXISTS genres (
    genre_id INTEGER PRIMARY KEY AUTOINCREMENT,
    genre_name TEXT NOT NULL UNIQUE
);

CREATE TABLE IF NOT EXISTS book_genres (
    book_id INTEGER,
    genre_id INTEGER,
    PRIMARY KEY (book_id, genre_id),
    FOREIGN KEY (book_id) REFERENCES books(book_id),
    FOREIGN KEY (genre_id) REFERENCES genres(genre_id)
);
""")
conn.commit()

### Load data from the existing pandas pipeline

This assumes `valid` (books filtered to a reliable year range, one row per book) and `genre_only` (exploded book-genre pairs, fiction/non-fiction excluded from the genre label set) already exist from the earlier EDA notebook. Adjust the `author` column name below if it differs in your dataframe.

In [ ]:
# 1. Authors — unique names get their own rows
authors_df = valid[["author"]].drop_duplicates().dropna()
authors_df.columns = ["author_name"]
authors_df.to_sql("authors_staging", conn, if_exists="replace", index=False)
cur.execute("""
    INSERT OR IGNORE INTO authors (author_name)
    SELECT DISTINCT author_name FROM authors_staging;
""")

In [1]:
print(len(authors_df))
print(valid.shape)

NameError: name 'authors_df' is not defined

In [ ]:
# 2. Genres — unique genre labels
genres_df = pd.DataFrame({"genre_name": genre_only["genres_list"].unique()})
genres_df.to_sql("genres_staging", conn, if_exists="replace", index=False)
cur.execute("""
    INSERT OR IGNORE INTO genres (genre_name)
    SELECT DISTINCT genre_name FROM genres_staging;
""")

In [ ]:
# 3. Books — one row per unique book, linked to author_id via lookup
books_df = valid.drop_duplicates(subset=["name"])[
    ["name", "author", "pub_year", "star_rating", "num_ratings", "isbn_clean"]
]
books_df.to_sql("books_staging", conn, if_exists="replace", index=False)
cur.execute("""
    INSERT INTO books (name, author_id, pub_year, star_rating, num_ratings, isbn_clean)
    SELECT bs.name, a.author_id, bs.pub_year, bs.star_rating, bs.num_ratings, bs.isbn_clean
    FROM books_staging bs
    LEFT JOIN authors a ON a.author_name = bs.author;
""")

In [ ]:
# 4. Book_genres — junction rows, mapped through the id lookups
bg_df = genre_only[["name", "genres_list"]].drop_duplicates()
bg_df.to_sql("bg_staging", conn, if_exists="replace", index=False)
cur.execute("""
    INSERT OR IGNORE INTO book_genres (book_id, genre_id)
    SELECT b.book_id, g.genre_id
    FROM bg_staging s
    JOIN books b ON b.name = s.name
    JOIN genres g ON g.genre_name = s.genres_list;
""")

conn.commit()

In [ ]:
# Clean up staging tables
cur.executescript("""
DROP TABLE authors_staging;
DROP TABLE genres_staging;
DROP TABLE books_staging;
DROP TABLE bg_staging;
""")
conn.commit()

In [ ]:
# Quick sanity check on row counts
for table in ["authors", "books", "genres", "book_genres"]:
    count = cur.execute(f"SELECT COUNT(*) FROM {table}").fetchone()[0]
    print(f"{table}: {count:,} rows")

## 4. Intermediate / Advanced SQL Queries

Covers: joins, grouping/aggregation, `HAVING`, and a subquery.

### a) JOIN + GROUP BY — most prevalent genre per year

In [ ]:
query_a = """
SELECT b.pub_year, g.genre_name, COUNT(DISTINCT b.book_id) AS book_count
FROM books b
JOIN book_genres bg ON b.book_id = bg.book_id
JOIN genres g ON bg.genre_id = g.genre_id
WHERE b.pub_year BETWEEN 1900 AND 2016
GROUP BY b.pub_year, g.genre_name
ORDER BY b.pub_year, book_count DESC;
"""
pd.read_sql(query_a, conn).head(20)

### b) GROUP BY + HAVING — genres with a meaningful footprint (filters out noise genres)

In [ ]:
query_b = """
SELECT g.genre_name, COUNT(DISTINCT bg.book_id) AS total_books
FROM genres g
JOIN book_genres bg ON g.genre_id = bg.genre_id
GROUP BY g.genre_name
HAVING COUNT(DISTINCT bg.book_id) > 1000
ORDER BY total_books DESC;
"""
pd.read_sql(query_b, conn).head(20)

### c) Subquery — authors whose average rating beats the overall dataset average

In [ ]:
query_c = """
SELECT a.author_name, AVG(b.star_rating) AS avg_rating, COUNT(b.book_id) AS num_books
FROM authors a
JOIN books b ON a.author_id = b.author_id
GROUP BY a.author_name
HAVING AVG(b.star_rating) > (
    SELECT AVG(star_rating) FROM books
)
ORDER BY avg_rating DESC
LIMIT 20;
"""
pd.read_sql(query_c, conn)

### d) Self-join-style co-occurrence — genres most often paired with "history"

This reproduces, in SQL, the History co-tag finding from the pandas EDA (History overlapping heavily with Biography, Politics, Philosophy, Religion, War, etc.), validating the same insight through a second method.

In [ ]:
query_d = """
SELECT g2.genre_name, COUNT(*) AS co_occurrences
FROM book_genres bg1
JOIN book_genres bg2 ON bg1.book_id = bg2.book_id AND bg1.genre_id != bg2.genre_id
JOIN genres g1 ON bg1.genre_id = g1.genre_id
JOIN genres g2 ON bg2.genre_id = g2.genre_id
WHERE g1.genre_name = 'history'
GROUP BY g2.genre_name
ORDER BY co_occurrences DESC
LIMIT 15;
"""
pd.read_sql(query_d, conn)

In [ ]:
conn.close()

In [2]:
import sqlite3
import pandas as pd
conn = sqlite3.connect("goodreads.db")
cur = conn.cursor()
cur.execute("PRAGMA foreign_keys = ON;")

In [3]:
from pathlib import Path
import pyarrow.parquet as pq
import pyarrow as pa
import numpy as np

folder = Path("Data/Goodreads_Books/genres_top100")
keep_cols = ["name", "author", "genres", "pub_year", "star_rating", "num_ratings", "isbn_clean"]

tables = []
for file in folder.glob("*.parquet"):
    table = pq.read_table(file, columns=keep_cols)
    genre_col = pa.array([file.stem] * table.num_rows)
    table = table.append_column("source_genre", genre_col)
    tables.append(table)

combined_table = pa.concat_tables(tables, promote_options="default")
df = combined_table.to_pandas()

df["pub_year"] = pd.to_numeric(df["pub_year"], errors="coerce", downcast="integer")
df["source_genre"] = df["source_genre"].astype("category")

valid = df[(df["pub_year"] >= 1900) & (df["pub_year"] <= 2016)].copy()

valid["genres_list"] = valid["genres"].apply(
    lambda g: [str(x).strip() for x in g] if isinstance(g, (list, tuple, np.ndarray)) else []
)

exploded = valid.explode("genres_list")
exploded = exploded[exploded["genres_list"].notna() & (exploded["genres_list"] != "")]
exploded["genres_list"] = exploded["genres_list"].str.strip().str.lower()

fiction_labels = {"fiction", "non-fiction", "nonfiction"}
genre_only = exploded[~exploded["genres_list"].isin(fiction_labels)]

print(valid.shape, genre_only.shape)

(4352106, 9) (24100598, 9)


In [ ]:
authors_df = valid[["author"]].drop_duplicates().dropna()
print(len(authors_df))

In [1]:
from pathlib import Path
folder = Path("Data/Goodreads_Books/genres_top100")
files = list(folder.glob("*.parquet"))
print(len(files), "files found")
print(files[:5])

100 files found
[WindowsPath('Data/Goodreads_Books/genres_top100/action.parquet'), WindowsPath('Data/Goodreads_Books/genres_top100/adult.parquet'), WindowsPath('Data/Goodreads_Books/genres_top100/adventure.parquet'), WindowsPath('Data/Goodreads_Books/genres_top100/amazon.parquet'), WindowsPath('Data/Goodreads_Books/genres_top100/american_history.parquet')]
